# Algoritmo de Simon (3 bits)

Este notebook implementa o **Algoritmo de Simon** para uma string binária secreta $s$ de 3 bits informada pelo usuário via prompt.

### Formulação Matemática
Dada uma função oráculo $f: \{0, 1\}^n \to \{0, 1\}^n$ com a promessa de que existe uma string oculta $s \in \{0, 1\}^n$ tal que, para quaisquer entradas $a, b \in \{0, 1\}^n$:
$$f(a) = f(b) \iff a \oplus b \in \{0^n, x\}$$

* Se $s = 000$, $f$ é **1-para-1** (bijetora).
* Se $s \neq 000$, $f$ é **2-para-1** com periodicidade $f(a) = f(a \oplus s)$.

O circuito quântico aplica o oráculo $U_f$, mede explicitamente o registrador de saída $|f(a)\rangle$ (colapsando o primeiro registrador para a superposição $\frac{1}{\sqrt{2}}(|a\rangle + |a \oplus s\rangle)$) e em seguida aplica $H^{\otimes n}$ e mede o primeiro registrador, obtendo vetores $y$ tais que $y \cdot s = 0 \pmod 2$.

In [2]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_aer import AerSimulator

ModuleNotFoundError: No module named 'qiskit_aer'

### 1. Entrada da String Binária Oculta ($x$)
Insira uma string de 3 bits (ex.: `110`, `101`, `011`, `000`).

In [ ]:
s_secret = '100'

### 2. Definição da Função Binária $f: \{0,1\}^3 \to \{0,1\}^3$ (Oráculo de Simon)
Construímos o operador unitário $U_f |a\rangle |0\rangle = |a\rangle |f(a)\rangle$ usando 6 qubits (3 para a entrada e 3 para a saída):
1. **Cópia inicial:** Aplicamos portas CNOT de cada qubit de entrada $a_i$ para o respectivo qubit de saída $b_i$, fazendo $f(a) = a$.
2. **Ocultação de $x$:** Se $x \neq 000$, escolhemos o índice $k$ do primeiro bit igual a `1` em $x$. Controlando no qubit $a_k$, aplicamos portas CNOT nos qubits de saída onde $x_i = 1$. Assim:
$$f(a) = a \oplus (a_k \cdot x) \implies f(a \oplus x) = f(a)$$

In [ ]:
def simon_oracle(x: str) -> QuantumCircuit:
    """
    Define o circuito oráculo U_f para uma função binária de 3 bits em 3 bits
    que oculta a string x (f(a) = f(a ^ x)).
    """
    n = len(x)
    oracle = QuantumCircuit(2 * n, name=f"U_f (x={x})")
    
    # Inverte a string para alinhar com a ordenação little-endian do Qiskit (qubit 0 = bit à direita)
    x_rev = x[::-1]
    
    # 1. Copia a entrada |a> para o registrador de saída: |a>|0> -> |a>|a>
    for i in range(n):
        oracle.cx(i, n + i)
        
    # 2. Se x != 000, garante f(a) = f(a ^ x)
    if '1' in x_rev:
        pivot_idx = x_rev.index('1')
        for i in range(n):
            if x_rev[i] == '1':
                oracle.cx(pivot_idx, n + i)
                
    return oracle

oracle_qc = simon_oracle(s_secret)
print("Circuito do Oráculo U_f:")
print(oracle_qc.draw('text'))

Circuito do Oráculo U_f:
                         
q_0: ──■─────────────────
       │                 
q_1: ──┼────■────────────
       │    │            
q_2: ──┼────┼────■────■──
     ┌─┴─┐  │    │    │  
q_3: ┤ X ├──┼────┼────┼──
     └───┘┌─┴─┐  │    │  
q_4: ─────┤ X ├──┼────┼──
          └───┘┌─┴─┐┌─┴─┐
q_5: ──────────┤ X ├┤ X ├
               └───┘└───┘


### 3. Execução do Circuito Quântico de Simon (com Medição após o Oráculo)
O circuito inclui **ambas as medições** padrão do Algoritmo de Simon:
1. **Medição do segundo registrador ($|f(a)\rangle$) logo após $U_f$:** projeta o estado no subespaço correspondente a um valor específico $f(a_0)$, deixando o primeiro registrador na superposição $\frac{1}{\sqrt{2}}(|a_0\rangle + |a_0 \oplus x\rangle)$.
2. **Transformada de Hadamard ($H^{\otimes 3}$) e medição do primeiro registrador:** produz strings $y$ ortogonais a $x$ em $\mathbb{F}_2$ ($y \cdot x = 0 \pmod 2$).

In [ ]:
n = len(s_secret)

# Registradores quânticos e clássicos separados para cada etapa de medição
qr_in = QuantumRegister(n, name='x_reg')
qr_out = QuantumRegister(n, name='fx_reg')
cr_in = ClassicalRegister(n, name='c_y')      # Medição final do registrador de entrada (y)
cr_out = ClassicalRegister(n, name='c_fx')    # Medição intermediária após o oráculo f(a)

qc = QuantumCircuit(qr_in, qr_out, cr_in, cr_out)

# 1. Superposição uniforme no registrador de entrada
qc.h(qr_in)
qc.barrier()

# 2. Consulta ao oráculo f: |a>|0> -> |a>|f(a)>
qc.compose(simon_oracle(s_secret), qubits=[*qr_in, *qr_out], inplace=True)
qc.barrier()

# 3. Medição do registrador auxiliar |f(a)> imediatamente após o oráculo
qc.measure(qr_out, cr_out)
qc.barrier()

# 4. Interferência via Transformada de Hadamard no registrador de entrada
qc.h(qr_in)
qc.barrier()

# 5. Medição final do registrador de entrada
qc.measure(qr_in, cr_in)

print("Circuito Completo de Simon:")
print(qc.draw('text'))

# Simulação quântica
simulator = AerSimulator()
compiled_qc = transpile(qc, simulator)
result = simulator.run(compiled_qc, shots=1024).result()
raw_counts = result.get_counts()

# Agrega as contagens marginalizando a medição intermediária c_fx (formato do Qiskit: 'c_fx c_y')
y_counts = {}
print("\nAmostras completas (f(a) medido após oráculo | y medido ao final):")
for outcome, count in sorted(raw_counts.items()):
    fx_measured, y_measured = outcome.split(' ')
    y_counts[y_measured] = y_counts.get(y_measured, 0) + count
    inner_prod = sum(int(b1) * int(b2) for b1, b2 in zip(y_measured, s_secret)) % 2
    print(f"  f(a) = {fx_measured} | y = {y_measured} -> {count:4d} ocorrências | y · x mod 2 = {inner_prod}")

Circuito Completo de Simon:
          ┌───┐ ░                      ░           ░ ┌───┐ ░ ┌─┐      
 x_reg_0: ┤ H ├─░───■──────────────────░───────────░─┤ H ├─░─┤M├──────
          ├───┤ ░   │                  ░           ░ ├───┤ ░ └╥┘┌─┐   
 x_reg_1: ┤ H ├─░───┼────■─────────────░───────────░─┤ H ├─░──╫─┤M├───
          ├───┤ ░   │    │             ░           ░ ├───┤ ░  ║ └╥┘┌─┐
 x_reg_2: ┤ H ├─░───┼────┼────■────■───░───────────░─┤ H ├─░──╫──╫─┤M├
          └───┘ ░ ┌─┴─┐  │    │    │   ░ ┌─┐       ░ └───┘ ░  ║  ║ └╥┘
fx_reg_0: ──────░─┤ X ├──┼────┼────┼───░─┤M├───────░───────░──╫──╫──╫─
                ░ └───┘┌─┴─┐  │    │   ░ └╥┘┌─┐    ░       ░  ║  ║  ║ 
fx_reg_1: ──────░──────┤ X ├──┼────┼───░──╫─┤M├────░───────░──╫──╫──╫─
                ░      └───┘┌─┴─┐┌─┴─┐ ░  ║ └╥┘┌─┐ ░       ░  ║  ║  ║ 
fx_reg_2: ──────░───────────┤ X ├┤ X ├─░──╫──╫─┤M├─░───────░──╫──╫──╫─
                ░           └───┘└───┘ ░  ║  ║ └╥┘ ░       ░  ║  ║  ║ 
   c_y: 3/════════════════════════════════╬══╬══╬

### 4. Pós-processamento Clássico (Eliminação Gaussiana em $\mathbb{F}_2$)
Resolvemos o sistema homogêneo $y \cdot x = 0 \pmod 2$ a partir das strings $y$ linearmente independentes obtidas na medição final para determinar a string oculta $x$.

In [ ]:
def recover_hidden_string(counts: dict, n: int) -> str:
    """
    Recupera a string oculta x de n bits resolvendo o sistema linear
    y · x = 0 (mod 2) via eliminação gaussiana em GF(2).
    """
    # Filtra medições não-nulas ordenadas pela frequência
    candidates = [y for y in sorted(counts, key=counts.get, reverse=True) if y != '0' * n]
    print(candidates)
    
    # Forma escalonada reduzida (RREF) em GF(2)
    basis = []  # lista de tuplas (coluna_pivo, vetor)
    for y_str in candidates:
        vec = [int(bit) for bit in y_str]
        
        for pivot_col, b_vec in basis:
            if vec[pivot_col] == 1:
                vec = [v ^ b for v, b in zip(vec, b_vec)]
                
        if 1 in vec:
            pivot_col = vec.index(1)
            for idx in range(len(basis)):
                p_col, b_vec = basis[idx]
                if b_vec[pivot_col] == 1:
                    basis[idx] = (p_col, [b ^ v for b, v in zip(b_vec, vec)])
            basis.append((pivot_col, vec))
            
        if len(basis) == n - 1:
            break

    # Caso f seja 1-para-1 (posto < n - 1), a string oculta é 000
    if len(basis) < n - 1:
        return '0' * n

    # Identifica a coluna livre e resolve para a solução não-trivial
    pivot_cols = {p_col for p_col, _ in basis}
    free_col = [c for c in range(n) if c not in pivot_cols][0]

    x_recovered = [0] * n
    x_recovered[free_col] = 1
    for p_col, b_vec in basis:
        x_recovered[p_col] = b_vec[free_col]

    return ''.join(str(bit) for bit in x_recovered)

x_found = recover_hidden_string(y_counts, n)

print("=" * 42)
print(f"String x fornecida  : {s_secret}")
print(f"String x encontrada : {x_found}")
print("=" * 42)
assert x_found == s_secret, "Divergência na recuperação da string!"


NameError: name 'y_counts' is not defined